# Evaluating Agents: measuring ability, duration, and value


> The previous lectures turned an LLM, step by step, into an Agent that can think, call tools, and plan: lecture 08 had the Agent do deep research, lecture 10 had it fix bugs in a real codebase, and lecture 11 added memory across sessions. The toolbox is already large, but one basic issue has not been faced directly: how to tell whether an Agent is doing well.
>
> This lecture builds an evaluation system. We first look at the limits of traditional benchmarks, then fill in three components in turn: METR uses human completion time as a yardstick, GDPval uses real economic tasks and blind review, and DeepScholar-Bench makes automatic scoring reliable enough to use. Finally we assemble the three into a minimal evaluation harness.

Place both models on a 100-item multiple-choice exam: one scores 0.97, the other 0.96 — a gap of one item, inside random fluctuation.

Give the same two models 8-hour long-horizon tasks, and the success rates become 0.02 versus 0.30, a factor of 15.

A score on a fixed exam cannot show the real gap between Agents. **Multiple-choice scores** put the two models on the same horizontal line; the long-task curve is what pulls the distance open.

To evaluate an Agent we prepare a batch of items with a clear right or wrong, let the Agent run them one by one, and count how many it got right. That fixed set of items is a **benchmark**, and the rule for counting an item as correct is the **pass criterion**. Split an evaluation open and it always answers three questions: where the tasks come from, what counts as correct, and who assigns the score. In the perceive–decide–act–feedback loop of lecture 1, this lecture fills in the feedback piece — without evaluation, however busy the Agent looks, there is no way to know whether it actually works. An Agent needs its own **evaluation**.

Start from the first layer: why a score on a fixed exam is not enough to evaluate an Agent.

The introduction closed on a claim: a score on a fixed exam is not enough to evaluate an Agent. This section explains why. Once the reasons a score cannot be trusted are clear, the next two sections' change of yardstick follows. In real development, choosing a model and comparing versions both rest on scores; whether those scores can be trusted decides whether those judgments can be trusted.

A multiple-choice score on a fixed exam has three layers of trouble for an Agent.

The first layer is saturation. Items such as HellaSwag are chosen on purpose, picking sentences a model is likely to get wrong. After models approach a perfect score, the remaining gaps are tiny, and the items no longer separate who is strong from who is weak. That state, "everyone's score is near the ceiling," is called saturation.

The second layer is the lack of a shared yardstick. Scores on different benchmarks are not comparable. A model near a perfect score on MMLU and struggling on SWE-bench cannot, from those two numbers, be translated into what it can actually do. Each benchmark has its own difficulty and scoring method, and there is no conversion between scores.

The third layer is that the tasks are not real. A multiple-choice exam has a fixed task distribution and a fixed scoring method, far from real work. Real work is not a sequence of independent items; it is one connected process.

The three layers point to the same fact: a multiple-choice score cannot be translated into "how many hours of work can be finished." That is especially clear on long tasks. A toy example first shows how a score can hide a difference in ability.

In [ ]:
# Nearby scores on a fixed exam, tenfold gaps on long-task success: the score hides the ability gap
import numpy as np

np.random.seed(42)
# Simulate 100 multiple-choice items: both models are near a perfect score
exam_a = np.random.binomial(n=1, p=0.97, size=100).mean()
exam_b = np.random.binomial(n=1, p=0.96, size=100).mean()

# Simulate long tasks: bucketed by human duration, model B is stronger on long tasks
durations = np.array([0.5, 1.0, 2.0, 4.0, 8.0])      # unit: hours
rate_a = np.array([0.90, 0.70, 0.40, 0.10, 0.02])
rate_b = np.array([0.85, 0.78, 0.65, 0.45, 0.30])

print("multiple-choice score: model A=%.3f  model B=%.3f" % (exam_a, exam_b))
print("8-hour task success rate: model A=%.2f  model B=%.2f" % (rate_a[-1], rate_b[-1]))
print("Key observation: the multiple-choice scores show no gap; on the long-task axis the two models separate at once.")


Read these numbers. The two models' multiple-choice scores are about 0.97 and 0.96 — a gap of one item in 100, inside random fluctuation. On 8-hour tasks the success rates are 0.02 versus 0.30, a factor of 15. The multiple-choice scores put the two models on the same horizontal line; the long-task curve pulls the distance open. The difference does not come from luck; it comes from the task: multiple-choice items are independent, short, and only right or wrong; a long task has layered dependencies, and any one failure can void the whole stretch of work.

The difference between model A and model B is hidden in the difficulty curve. On short tasks both are near 0.9; as the task lengthens, model A's success rate falls quickly and model B's falls more slowly. With task duration as the horizontal axis, the ability gap is clear on the curve, which is the yardstick the next section introduces.

## 2. Measuring task difficulty by completion time


The previous section's example showed that without a shared yardstick, scores cannot separate strong models from weak ones. This section introduces a yardstick that does not depend on any one set of items: how long a human needs to finish a task. The longer the tasks a model can complete, the stronger it is. This "time as yardstick" scale is used in real development both to compare models sideways and to watch how much an in-house model strengthens every few months.

METR puts the time yardstick on Agents. It builds a suite of software and machine-learning tasks, with difficulty from a few seconds to 8 hours, has human experts complete each task, and takes the geometric mean of completion times as the task's difficulty t. Why the geometric mean rather than the ordinary mean is clearer from an example. Suppose one task takes humans 1 minute on average and another takes 100 minutes. The arithmetic mean is about 50 minutes; the geometric mean is $\sqrt{1\times 100}=10$ minutes. Long tasks pull the arithmetic mean toward the large end; the geometric mean sits in the middle of the log scale and better represents the duration of a typical task. The geometric mean is defined as $t=\exp\big(\tfrac{1}{N}\sum_i \log t_i\big)$, that is, take logs, average, then exponentiate.

Tasks come from three blocks: HCAST covers ordinary software tasks from 1 minute to 30 hours, RE-Bench is 8-hour machine-learning research tasks, and SW AA is 1-to-30-second single-step micro-tasks.

Each Agent is run several times on each task, yielding a success rate. Put task difficulty t and success rate p together and fit a logistic curve:

$$p(\text{task}) = \sigma\!\big((\log h - \log t)\cdot \beta\big)$$

In the formula, h is the model's 50% time horizon, the human duration at which it succeeds with probability one half; beta controls how steep the curve is. First unpack the term in parentheses. $\log h - \log t = \log(h/t)$, which depends only on the ratio of h to t, not on their absolute sizes. That is the property long-task evaluation needs: task duration from 1 second to 100 hours spans 5 orders of magnitude, addition and subtraction on the raw scale are meaningless, and "a factor of a few" is what matters. Taking the natural log of t compresses each 10-fold span into a cell of equal width: $\ln 2\approx 0.69$, $\ln 20\approx 3.00$, $\ln 200\approx 5.30$, and adjacent cells differ by $\ln 10\approx 2.30$.

The outer $\sigma$ is the logistic function $\sigma(z)=1/(1+e^{-z})$. When z is large, $e^{-z}$ approaches 0 and $\sigma$ approaches 1; when z is small, $\sigma$ approaches 0; at z=0, $\sigma=0.5$. It compresses an input from negative infinity to positive infinity into 0 to 1, which is exactly the range of a probability.

When h equals the task duration t, $\log h - \log t = 0$, $\sigma(0) = 0.5$, and the success rate is exactly one half. First compute this curve by hand.

A hand calculation for the numbers in the code below, using the same values: $h=2$ hours, $\beta=2$, substituted into three task durations t.

First, $t=2$ hours, $h=t$:
$$\log h - \log t = \log 2 - \log 2 = 0,\quad z = 0\times 2 = 0,\quad p=\sigma(0)=0.5$$
When the task duration equals the time horizon, the model succeeds with probability one half — that is where the name 50% time horizon comes from.

Second, $t=8$ hours, the task is 4 times the time horizon:
$$\log h - \log t = \log(2/8)=\log 0.25\approx -1.386,\quad z = -1.386\times 2\approx -2.77$$
$$e^{-z}=e^{2.77}\approx 16,\quad p=\frac{1}{1+e^{-z}}\approx\frac{1}{17}\approx 0.059$$
The 8-hour success rate is about 0.06. Without a table, the order of magnitude is still clear: $e^{2.77}$ is about 16, so the success rate is about $1/17$.

Third, $t=0.5$ hours, the task is 4 times shorter than the time horizon:
$$\log h - \log t = \log(2/0.5)=\log 4\approx 1.386,\quad z=1.386\times 2\approx 2.77$$
$$e^{-z}=e^{-2.77}\approx 0.0625=\frac{1}{16},\quad p=\frac{1}{1+0.0625}\approx 0.941$$
The half-hour success rate is about 0.94.

Now look at beta on its own. Keep $t=8$ hours and $h=2$ hours, and change beta from 2 to 0.5: $z=-1.386\times 0.5\approx -0.693$, $e^{-z}=2$, $p=1/3\approx 0.333$. On the same task, beta=2 gives a success rate of only 0.06, while beta=0.5 still has 0.33. Beta describes how fast ability falls as the task lengthens: the larger beta is, the steeper the curve near h, and a task only slightly harder than h slides quickly toward success rate 0; the smaller beta is, the flatter the curve, and some success remains even on harder tasks. On a log scale, $\log h - \log t$ is the distance between two points, and beta decides how that distance converts into a probability.

In [ ]:
# Hand calculation of the logistic: the relative position of h and t decides the success rate
import numpy as np

def sigma(z):
    """logistic sigmoid."""
    return 1.0 / (1.0 + np.exp(-z))

def success_prob(log_h, log_t, beta):
    """Success rate by the METR formula."""
    return sigma((log_h - log_t) * beta)

beta = 2.0
log_h = np.log(2.0)      # this model's 50% time horizon h = 2 hours

print("h=t=2h    : p = %.3f  (should be near 0.5)" % success_prob(log_h, np.log(2.0), beta))
print("t=8h (hard): p = %.3f" % success_prob(log_h, np.log(8.0), beta))
print("t=0.5h (easy): p = %.3f" % success_prob(log_h, np.log(0.5), beta))
print("Key observation: p=0.5 when h=t; longer tasks have lower success; beta controls the rate of decline.")


In [ ]:
# Synthetic "task duration vs success rate" data; fit the 50% time horizon with numpy
import numpy as np
from scipy.optimize import minimize

def neg_log_likelihood(params, log_t, y):
    """Negative log-likelihood: params=(log h, beta), y is 0/1 success or failure."""
    log_h, beta = params
    p = 1.0 / (1.0 + np.exp(-(log_h - log_t) * beta))
    eps = 1e-12
    return -np.sum(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

np.random.seed(42)
# 60 tasks, human duration drawn log-uniform, covering 1 minute to 8 hours
log_t = np.random.uniform(np.log(1.0 / 60.0), np.log(8.0), size=60)
true_log_h = np.log(2.0)            # true horizon 2 hours
true_beta = 2.5
p_true = 1.0 / (1.0 + np.exp(-(true_log_h - log_t) * true_beta))
y = np.random.binomial(n=1, p=p_true, size=60)      # one success/failure sample per task

# Fit: minimize the negative log-likelihood, with a reasonable range for log h and beta
init = np.array([np.log(1.0), 1.0])
res = minimize(neg_log_likelihood, init, args=(log_t, y),
               method="L-BFGS-B", bounds=[(-10.0, 10.0), (0.05, 10.0)])
log_h_est, beta_est = res.x

print("true 50%% horizon h=2.0 hours, fitted h=%.2f hours" % np.exp(log_h_est))
print("fitted beta=%.2f (true %.2f)" % (beta_est, true_beta))
print("Key observation: task duration plus a success/failure record is enough to recover the model's ability yardstick.")


In [ ]:
# Ability–duration curve: horizontal axis is the human duration of the task, vertical axis is success rate
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
grid_t = np.logspace(np.log10(1.0 / 60.0), np.log10(8.0), 200)
p_fit = 1.0 / (1.0 + np.exp(-(log_h_est - np.log(grid_t)) * beta_est))

plt.figure(figsize=(6, 4))
plt.scatter(np.exp(log_t), y + 0.02 * np.random.uniform(-1, 1, len(y)),
            s=20, alpha=0.6, label="observed tasks")
plt.plot(grid_t, p_fit, color="tab:red", label="fitted curve")
plt.axvline(np.exp(log_h_est), color="gray", linestyle="--")
plt.text(np.exp(log_h_est) * 1.6, 0.55, "h=%.1fh" % np.exp(log_h_est),
         color="gray", fontsize=9)
plt.xscale("log")
plt.xlabel("human task duration (hours)")
plt.ylabel("success rate")
plt.ylim(-0.05, 1.05)
plt.legend()
plt.tight_layout()
plt.show()
print("Key observation: the fitted curve crosses 50% at the horizontal coordinate that is the time horizon.")


A single time horizon is a snapshot; more informative is how it changes with release date. METR fits a 50% time horizon for each of a batch of models, then runs a linear regression of log h on release date and converts the slope into "doubling days." The conclusion is that frontier models' horizon roughly doubles every 207 days, about 7 months: GPT-2 has 2 seconds, o3 about 110 minutes.

The slope of the trend line is more trustworthy than the height of any single model. Errors of different models on the same task suite are highly correlated, so comparing two models' absolute horizons on their own is unreliable; the regression slope absorbs those correlated errors together.

The conversion to doubling days is worth computing by hand. The regression slope is how much log h increases per month. Doubling the horizon means log h increases by $\ln 2\approx 0.693$, so the number of months to double is $0.693\div$ slope, then times 30 is days. Suppose a model has a 0.5-hour horizon in January 2019 and 1 hour six months later; the slope is $\ln(1/0.5)/6\approx 0.116$ per month, and doubling days $=0.693/0.116\times 30\approx 180$ days. The code below fits about 182 days on synthetic data, the same order of magnitude as the 207 days METR reports on real data.

In [ ]:
# Linear regression of log h on release date; convert the slope into doubling days
import numpy as np
import matplotlib.pyplot as plt

# Synthetic release months (from January 2019) and corresponding 50% time horizons for 8 models
months_since = np.array([0, 6, 12, 24, 36, 48, 60, 72])
horizons = np.array([0.0006, 0.0015, 0.004, 0.01, 0.05, 0.25, 1.0, 1.8])

log_h = np.log(horizons)
A = np.column_stack([np.ones_like(months_since), months_since])
coef, *_ = np.linalg.lstsq(A, log_h, rcond=None)
intercept, slope = coef

doubling_days = np.log(2.0) / slope * 30.0       # convert months at 30 days each

plt.figure(figsize=(6, 4))
plt.scatter(months_since, log_h, s=30, label="models")
xs = np.array([0, 72])
plt.plot(xs, intercept + slope * xs, color="tab:red", label="OLS fit")
plt.xlabel("months since 2019-01")
plt.ylabel("log 50% horizon (hours)")
plt.legend()
plt.tight_layout()
plt.show()

print("doubling days ≈ %.0f days (METR reports about 207 days)" % doubling_days)
print("Key observation: the slope of a straight line on a log scale compresses ability progress into one number.")


## 3. Comparing two Agents by blind review


The time horizon answers "how long a task can be completed"; it does not answer "how much those tasks are worth, or how good the completion is." GDPval changes the yardstick: it measures real economic tasks. From the 9 industries that contribute most to GDP, it selects the 5 highest-paid digital occupations in each industry, 44 occupations in total. Occupation experts turn real products of their daily work into evaluation tasks — each task is a request plus a deliverable, taking about 9.5 hours on average, with dollar value converted as "mean duration × occupational median hourly wage," about 398 dollars on average.

The scoring method is blind review: the model's deliverable and a human expert's deliverable are paired, anonymized, and given to an expert in that occupation to decide which is better. "Blind" means the reviewer does not know which deliverable came from the model and which from the expert. If identity were told to the reviewer, the model's prior reputation and labels of writing style would mix into the score — the same text, tagged "AI-generated" versus "expert-written," can receive different scores. Blind review isolates identity from scoring and leaves only the content.

Paired comparison rather than a single score also has a purpose. Absolute scoring (give this deliverable a 7) depends on each reviewer's internal scale, which is hard to align across reviewers; paired comparison only asks which of the two is better, a judgment closer to real use and more stable. The score takes three values: model wins is 1, tie is 0.5, model loses is 0. A tie counted as 0.5 means the model did not beat the human and also did not lose, and in the win rate it converts to half a win. The win rate is the mean of these scores, which also equals "wins plus half the ties" over the number of rounds, that is $(\text{wins}\times 1 + \text{ties}\times 0.5)/N$. First compute a paired-comparison table by hand.

Compute the win rate by hand on 10 tasks. Each task is one paired comparison, with result in {win, tie, loss}:

| task | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|:---|:---|:---|:---|:---|:---|:---|:---|:---|:---|:---|
| model vs expert | win | win | tie | loss | win | tie | win | loss | tie | win |

Count the three values: 5 wins, 3 ties, 2 losses. The win rate is the mean score:

$$\text{win rate}=\frac{5\times 1+3\times 0.5+2\times 0}{10}=\frac{6.5}{10}=0.65$$

An equivalent algorithm is "wins plus half the ties" divided by the number of rounds: $(5+3/2)/10=0.65$; both algorithms give the same result. 0.65 means: drawing a task at random, the model's deliverable has a 65% chance of matching or beating the expert in blind review — a full win 50% of the time, a tie 15% of the time.

Ties and losses should be kept apart. If a tie is treated as a loss (scored 0), the win rate becomes 0.5; treated as a win (scored 1), it becomes 0.8. 0.65 is the neutral convention; it answers not "how many times the model won" but "how close the model's quality is to the expert." The code below also computes the "at or above human" rate $(5+3)/10=0.80$, counting every tie as meeting the bar, a looser convention. The two numbers together distinguish "the model often ties" from "the model truly wins." 

In [ ]:
# Hand calculation of blind-review win rate: three-value scores {win 1, tie 0.5, loss 0}
import numpy as np

# 10 tasks, each cell is the comparison result of "model vs human expert"
pairwise = np.array([1.0, 1.0, 0.5, 0.0, 1.0, 0.5, 1.0, 0.0, 0.5, 1.0])
wins = int(np.sum(pairwise == 1.0))
ties = int(np.sum(pairwise == 0.5))
losses = int(np.sum(pairwise == 0.0))

win_rate = np.mean(pairwise)                  # expected score with ties counted as 0.5
better_or_equal = (wins + ties) / len(pairwise)   # fraction at or above human

print("win=%d  tie=%d  loss=%d  (%d tasks)" % (wins, ties, losses, len(pairwise)))
print("win rate = %.2f" % win_rate)
print("at or above human = %.2f" % better_or_equal)
print("Key observation: counting a tie as 0.5 in the win rate is the conservative convention for \"quality approaching human.\"")


True expert blind review is expensive; a single paired comparison takes more than 1 hour. GDPval trains an automatic scorer (LLM judge) to replace the expert, but the scorer's reliability has to be measured first. Let A be the automatic scorer's 0/0.5/1 score on a task, H the human score; agreement is defined as

$$A^{\text{HA}} = E[\,1 - |H - A|\,]$$

Human–human agreement is $1 - |H_1 - H_2|$ between two human scores H1, H2 on the same task. The two numbers have to be read together: the automatic scorer's agreement with humans should not be much worse than humans' agreement with each other. In GDPval the two are 66% and 71%. A small dataset reproduces this structure.


The agreement formula $1-|H-A|$ can be computed task by task. When the two scores are identical, $|H-A|=0$, agreement 1; a difference of 0.5 (one win, one tie) gives agreement 0.5; a difference of 1 (one win, one loss) gives agreement 0. The average is a number between 0 and 1. The 10-task data below matches the arrays in the code cell; the automatic scorer's score A is compared task by task with the mean of two human experts:

| task | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|:---|:---|:---|:---|:---|:---|:---|:---|:---|:---|:---|
| A | 1 | 1 | 0.5 | 0 | 0.5 | 1 | 0 | 0 | 1 | 0.5 |
| (H1+H2)/2 | 1 | 0.5 | 0.5 | 0 | 0.5 | 1 | 0.5 | 0 | 1 | 0.75 |
| 1-|A-H| | 1 | 0.5 | 1 | 1 | 1 | 1 | 0.5 | 1 | 1 | 0.75 |

Sum term by term: $1+0.5+1+1+1+1+0.5+1+1+0.75=8.75$, divide by 10 to get $A^{\text{HA}}=0.875$.

Human–human agreement $A^{\text{HH}}$ is computed directly from H1 and H2: on 9 of 10 tasks the two humans score identically (agreement 1), and only on task 10 H1=1, H2=0.5 differ by 0.5 (agreement 0.5). The sum is $9\times 1+0.5=9.5$, divide by 10 to get 0.95. The automatic scorer's 0.875 is below the humans' 0.95, but the gap is small, which means this scorer is already close to the tacit agreement between two human experts.

Putting $A^{\text{HA}}$ and $A^{\text{HH}}$ side by side is the key. Human–human agreement is the ceiling reference an automatic scorer can reach — two humans are not 100% consistent either, so requiring the machine to agree with humans 100% is meaningless. As long as the automatic scorer's agreement with humans is not much worse than humans with each other, replacing the expert with a machine is acceptable. The 66% versus 71% that GDPval reports is this comparison on real data.

In [ ]:
# Agreement: automatic scorer vs human, human vs human
import numpy as np

# 10 tasks: automatic scorer scores A, two human experts score H1, H2
A  = np.array([1.0, 1.0, 0.5, 0.0, 0.5, 1.0, 0.0, 0.0, 1.0, 0.5])
H1 = np.array([1.0, 0.5, 0.5, 0.0, 0.5, 1.0, 0.5, 0.0, 1.0, 1.0])
H2 = np.array([1.0, 0.5, 0.5, 0.0, 0.5, 1.0, 0.5, 0.0, 1.0, 0.5])

def agreement(x, y):
    """Mean agreement 1 - |x - y| of two score sequences."""
    return np.mean(1.0 - np.abs(x - y))

A_HA = agreement(A, (H1 + H2) / 2.0)      # automatic scorer vs mean human score
A_HH = agreement(H1, H2)                   # humans with each other

print("A_HA (automatic scorer vs human) = %.2f" % A_HA)
print("A_HH (human–human) = %.2f" % A_HH)
print("Key observation: the automatic scorer's agreement with humans is slightly below human–human agreement; that gap has to be monitored.")


Blind review answers "how many rounds were won"; a real user cares more about "how much time handing the work to the model saves." A realistic workflow is "try the model first, then fix it oneself if unsatisfied." Let the model's win rate on a task be w, the model's completion time M_T, the expert review time R_T, and the human's own completion time H_T. Trying once, the model output is accepted as-is if acceptable (probability w), and redone from scratch if not (probability 1-w); the expected time is

$$E[T_{1,i}] = M_{T,i} + R_{T,i} + (1 - w_i)\, H_{T,i}$$

Trying n times, summing the geometric series gives:

$$E[T_{n,i}] = (M_{T,i}+R_{T,i})\,\frac{1-(1-w_i)^n}{w_i} + (1-w_i)^n\, H_{T,i}$$

First compute try-1 expected time at several win rates by hand, then look at the curve.


Unpack the formula first. The try-1 expected time $M_T+R_T+(1-w)H_T$ has two parts: whether the output is acceptable or not, the model still has to run once and the expert still has to review once, so that $M_T+R_T$ is always paid; only when the output is unacceptable (probability $1-w$) is the extra $H_T$ paid for a human to do it from scratch. Substitute the small numbers from the code: $M_T=0.5$ hours, $R_T=0.3$ hours, $H_T=2.0$ hours.

| win rate w | $M_T+R_T$ | $(1-w)H_T$ | expected total time | vs pure human 2.0h |
|:---|:---|:---|:---|:---|
| 0.2 | 0.8 | $0.8\times 2.0=1.6$ | 2.4 hours | slower |
| 0.5 | 0.8 | $0.5\times 2.0=1.0$ | 1.8 hours | slightly faster |
| 0.8 | 0.8 | $0.2\times 2.0=0.4$ | 1.2 hours | much faster |
| 1.0 | 0.8 | 0 | 0.8 hours | 2.5 times faster |

At w=0.2, model assistance is slower than pure human work: the model fails often, and rework cost eats the time saved. The gain from model assistance comes from the win rate, not from the model's own speed. Try-1 starts to beat pure human work only when w exceeds $(M_T+R_T)/H_T=0.8/2.0=0.4$.

The try-n formula can be derived by hand. The policy is to try at most n times, each with success probability w, and hand over to a human only if all fail. Each attempt pays $M_T+R_T$; the expected number of attempts is $1+q+q^2+\cdots+q^{n-1}$ (write $q=1-w$; the k-th attempt happens if and only if the first k-1 failed, probability $q^{k-1}$), and the geometric series sums to $(1-q^n)/(1-q)=(1-q^n)/w$. Only if all n attempts fail (probability $q^n$) is the extra $H_T$ paid. Putting the two terms together is $E[T_{n,i}]$. When n=1, $(1-q)/w=1$, and the formula reduces to the try-1 expression.

In [ ]:
# Hand calculation of try-1 expected cost: model assistance vs pure human
import numpy as np

M_T = 0.5     # model completion: 0.5 hours
R_T = 0.3     # expert review: 0.3 hours
H_T = 2.0     # human completing it: 2 hours

def expected_time_try1(w):
    """Try once: model + review; on failure a human redoes the work."""
    return M_T + R_T + (1 - w) * H_T

for w in [0.2, 0.5, 0.8, 1.0]:
    print("w=%.1f: expected %.2f hours (pure human %.1f hours)" % (w, expected_time_try1(w), H_T))

print("Key observation: at w=0.2 model assistance is slower; the gain depends entirely on a high enough win rate.")


In [ ]:
# try-n cost curve: scan win rate and number of attempts, find the crossover
import numpy as np
import matplotlib.pyplot as plt

M_T, R_T, H_T = 0.5, 0.3, 2.0

def expected_time_tryn(w, n):
    """Expected time of n attempts; when w is 0, reduce to pure human."""
    q = 1.0 - w
    if w <= 0:
        return H_T
    return (M_T + R_T) * (1 - q**n) / w + q**n * H_T

ws = np.linspace(0.0, 1.0, 100)
plt.figure(figsize=(6, 4))
for n in [1, 3, 10]:
    plt.plot(ws, [expected_time_tryn(w, n) for w in ws], label="try-%d" % n)
plt.axhline(H_T, color="gray", linestyle="--", label="human only")
plt.axvline((M_T + R_T) / H_T, color="gray", linestyle=":", label="crossover")
plt.xlabel("model win rate w")
plt.ylabel("expected time (hours)")
plt.ylim(0, 2.5)
plt.legend()
plt.tight_layout()
plt.show()
print("Key observation: model assistance starts to pay only after the win rate exceeds (M_T+R_T)/H_T=%.2f."
      % ((M_T + R_T) / H_T))


## 4. Why automatic scoring distorts


GDPval already uses an automatic scorer in place of human experts, but the scorer's own reliability still has to be calibrated. This section switches to a harder evaluation object: deep research. DeepScholar-Bench evaluates exactly that task, and two of its features make evaluation especially hard.

The first feature is that the task is open. Deep research has to read the literature, synthesize viewpoints, and write a survey; there is no unique correct answer, so it is an open long-horizon task, and every metric is scored by an LLM judge. The second feature is fear of item leakage. If the answer to an evaluation item is already in the model's training data, the model can score high by memorizing, and the score no longer speaks to ability. That leak of answers to the model is data leakage. To avoid it, DeepScholar-Bench uses a continually updated pipeline that auto-generates tasks from recently accepted arXiv papers — given a paper title and abstract, generate a related-work section, with the human authors' original text as the reference answer.

Evaluation has three dimensions. The first is knowledge synthesis, whether the survey is clearly organized and whether the key points are covered, with metrics organization and nugget coverage. The second is retrieval quality, whether cited sources are the right ones and whether important literature is missing, with metrics relevance rate, reference coverage, and document importance. The third is verifiability, whether claims in a sentence have a matching citation, with metrics citation precision and claim coverage.

Every metric has to be calibrated against human annotation first. DeepScholar-Bench aligns with 300-plus annotations from 11 PhDs; LLM-judge agreement with humans is about 71% on organization and about 83% on nugget labels. One of the most typical problems of automatic scoring is position bias; a demonstration first.

Position bias means: in the same comparison, the order in which answers appear can change the judge's conclusion. The reason is that an LLM often assigns higher weight to content that appears first, or attention decays toward the end. The code below uses a deliberately simplified rule judge — it unconditionally selects the answer that is listed first — to magnify position bias until it is visible at a glance.

A real LLM judge is not this extreme, but the direction is the same: a good answer listed first may be judged good, listed second may be judged worse. The standard check for whether a judge has position bias is swap averaging: score the same pair of answers in both orders; if the two conclusions disagree, the conclusion depends on position rather than content. When the two conclusions disagree, treat it as a tie; when they agree, keep the conclusion. In the code the good answer wins once and loses once, scores [1, 0], and the swap-average win rate is 0.5 — position bias is flattened into a neutral tie.

There are other practical defenses against position bias: shuffle the order, repeat, and average; steer the judge's attention onto the content itself; replace an open "which is better" with a strictly defined rubric. Position bias is not the judge's only problem. Papers in the same direction also observe that a judge prefers its own generations, called self-recognition bias. Position bias is the easiest to demonstrate, because it can be exposed directly by two comparisons in opposite order.

In [ ]:
# Position bias of an LLM judge: the same pair of answers, swapped order, may reverse the conclusion
import numpy as np

def judge_first(text):
    """Rule-simulated judge: unconditionally select the answer listed first."""
    pos_a = text.find("ANSWER_A")
    pos_b = text.find("ANSWER_B")
    return "A" if pos_a < pos_b else "B"

def judge_pair(a, b):
    """Score the same pair of answers in both orders; return the two winners."""
    order1 = judge_first("ANSWER_A=" + a + " ANSWER_B=" + b)
    order2 = judge_first("ANSWER_B=" + a + " ANSWER_A=" + b)
    return order1, order2

good = "Clear structure, covers the key points."
bad = "Scattered content, misses key points."
r1, r2 = judge_pair(good, bad)

print("order (good first): judge picks %s" % r1)
print("order (good second): judge picks %s" % r2)
print("Key observation: the same judge gives opposite conclusions from position alone; a single score cannot be trusted.")

# Swap average: when the two orders disagree, treat as a tie, flattening position bias
scores = [1.0, 0.0]        # good answer's scores in the two comparisons: one win, one loss
win_rate_swap = np.mean(scores)
print("good answer's two scores: %s, swap-average win rate = %.2f" % (scores, win_rate_swap))
print("Key observation: averaging the two orders flattens a directional bias into a tie and cancels position bias.")


The output of a research survey is a stretch of text plus a list of citations, and evaluation looks at three metrics. Relevance rate: each cited source is scored 0/1/2 by a judge, and RR is the mean score divided by 2. Reference coverage: of the important citations marked in the human exemplar, how many the report cites, which is recall of important citations. Verifiability: how many of the claims in sentences are supported by the citations attached to that sentence (claim coverage). The three metrics answer "are the citations right, are they complete, and do they hold up."

Compute the three metrics by hand with the numbers in the code below. First relevance: the judge's scores on 5 cited sources are $[2,1,2,0,2]$, mean $(2+1+2+0+2)/5=1.4$, divide by 2 to get RR=0.7. The divide by 2 is because the scoring cap is 2, scaling the 0–2 interval onto 0–1. Relevance answers whether cited sources are the right ones: a 0 means the source is off-topic, a 2 means it is directly relevant, and a higher RR means the report is not citing at random.

Reference coverage is recall of important citations. The human exemplar has 6 important citations, the report cites 4 of them, RC=4/6≈0.667. It does not care whether the report also cites irrelevant literature (that is relevance's job); it only cares whether what should be cited is missing. Verifiability (claim coverage) is counted at the sentence level: of the report's 8 sentences, 7 have their claim supported by the citations attached to that sentence, claim coverage=7/8=0.875.

| metric | question it answers | calculation | this toy example |
|:---|:---|:---|:---|
| relevance rate | are the citations right | mean score of cited sources $\div 2$ | 1.4/2 = 0.7 |
| reference coverage | are they complete | important citations hit $\div$ total important citations | 4/6 ≈ 0.667 |
| claim coverage | do they hold up | sentences with citation support $\div$ total sentences | 7/8 = 0.875 |

The three metrics have different denominators, but each is a recall-style fraction of some total, each landing between 0 and 1. When evaluating deep research, the three describe the same report from different angles, and any one that is low is worth checking on its own.

In [ ]:
# Toy retrieval-quality calculation: relevance RR, reference coverage RC, verifiability claim coverage
import numpy as np

# Relevance: the judge scores each cited source 0/1/2
rels = np.array([2, 1, 2, 0, 2])
RR = rels.mean() / 2.0

# Reference coverage: the exemplar has 6 important citations, the report cites 4
RC = 4 / 6

# Verifiability: of 8 sentences in the report, 7 have their claim supported by their citations
claim_cov = 7 / 8

print("Relevance Rate RR = %.3f (max is 1)" % RR)
print("Reference Coverage RC = %.3f (recall of important citations)" % RC)
print("Verifiability (claim coverage) = %.3f" % claim_cov)
print("Key observation: the three metrics answer whether citations are right, complete, and holding up.")


In [ ]:
# Oracle ablation and geometric mean: retrieval is not the bottleneck, synthesis is
import numpy as np

# Numbers reported by DeepScholar: feed the system "the literature the correct answer should cite"
base_rc = 0.187          # original reference coverage
oracle_rc = 1.0
base_nugget = 0.392      # original nugget coverage
oracle_nugget = 0.49

print("reference coverage: %.3f -> %.3f" % (base_rc, oracle_rc))
print("nugget coverage: %.3f -> %.3f" % (base_nugget, oracle_nugget))
print("Key observation: with retrieval all correct, nugget coverage rises only 0.1; the bottleneck is synthesis, not retrieval.")

def geomean(x):
    """Geometric mean; return 0 if any term is 0."""
    if np.any(x <= 0):
        return 0.0
    return np.exp(np.mean(np.log(x)))

system_c = np.array([0.95, 0.0, 0.95, 0.95, 0.95])
print("a system with one term 0: arithmetic mean %.3f, geometric mean %.3f"
      % (system_c.mean(), geomean(system_c)))
print("Key observation: the geometric mean punishes a short board; a single 0 sends the total to 0.")


The tools of the first three sections each solve one component: the time horizon handles "task difficulty spans a wide range," win rate and agreement handle "the criterion is not unique," and the LLM judge handles "scoring is automated," while its biases have to be guarded against. Putting them together is an evaluation harness — task environment, scoring function, and aggregate statistics each in place. A toy task suite walks through the full pipeline below.

The code below unfolds in three blocks: first define the toy task suite and tools (task environment), then write the agent and the scorer (scoring function), then run repeatedly and summarize (aggregate statistics). The toy tasks' scoring function reduces to a Boolean comparison "output equals the expected value," so attention stays on the harness skeleton rather than on scoring details.

**How it runs**: the agent below is driven by llm_client; under the scripted demo, deterministic scripted actions execute, and in live mode the model outputs actions. In both cases the harness's judgment logic is the same.

In [ ]:
# Unified LLM client: use a real model when an API key is present, otherwise fall back to a scripted demo
import sys, os
_root = os.path.abspath(os.getcwd())
while not os.path.exists(os.path.join(_root, 'llm_client.py')):
    _root = os.path.dirname(_root)
    if _root == os.path.dirname(_root):
        break
if _root not in sys.path:
    sys.path.insert(0, _root)
from llm_client import get_llm
client = get_llm()
print("LLM mode:", "scripted example (deterministic placeholder output)" if False else "real")


In [ ]:
# Evaluation harness, block 1: task environment (toy task suite, programmatically judgeable)
# Each task = (name, arguments, expected result); scoring is "whether the output equals the expected value"
TASKS = [
    ("sum", (3, 5), 8),
    ("sum", (12, 7), 19),
    ("difference", (10, 4), 6),
    ("difference", (20, 9), 11),
    ("product", (6, 7), 42),
    ("product", (4, 9), 36),
    ("negate", (9,), -9),
    ("square", (4,), 16),
]

# Tool set: add, subtract, multiply, negate, square
def tool_add(a, b):
    """Addition."""
    return a + b

def tool_sub(a, b):
    """Subtraction."""
    return a - b

def tool_mul(a, b):
    """Multiplication."""
    return a * b

def tool_neg(a):
    """Negation."""
    return -a

def tool_sq(a):
    """Square."""
    return a * a

TOOLS = {"add": tool_add, "sub": tool_sub, "mul": tool_mul,
         "neg": tool_neg, "sq": tool_sq}

# Block 2: agent. The LLM decides the action, the tool executes, the scoring function judges right or wrong
import re

def parse_action(text):
    """Parse Action: name(args) from LLM output; return None if parsing fails."""
    m = re.search(r"Action:\s*(\w+)\(([^)]*)\)", text)
    if not m:
        return None
    name = m.group(1)
    args = [int(x) for x in re.findall(r"-?\d+", m.group(2))]
    return name, args

def make_agent(client):
    """Build an "LLM decides + tool executes" agent, returning an act(task) function.

    The scripted demo returns scripted actions, using addition by mistake on product tasks to create an observable failure;
    live mode has the model output an Action line.
    """
    scripted_plan = {"sum": "add", "difference": "sub", "product": "add",
                 "negate": "neg", "square": "sq"}

    def act(task):
        name, args, _ = task
        if False:
            tool = scripted_plan[name]
            return "Action: %s(%s)" % (tool, ", ".join(map(str, args)))
        prompt = ("Task: %s, arguments %s. Output only one Action line, "
                  "for example Action: add(3, 5)." % (name, args))
        return client.chat([{"role": "user", "content": prompt}])

    return act

print("task suite %d items, tools %d, agent ready." % (len(TASKS), len(TOOLS)))


In [ ]:
# Block 3: run. Repeat each task N_RUNS times, record success/failure, duration, and the action path
import time
import numpy as np

np.random.seed(42)
act = make_agent(client)
N_RUNS = 3
rows = []

def run_once(act, task):
    """Run a single task once. Return (passed, output, path, duration in seconds)."""
    name, args, expected = task
    t0 = time.time()
    action_text = act(task)
    parsed = parse_action(action_text)
    if parsed is None:
        return False, "PARSE_FAIL", [("action", action_text)], time.time() - t0
    tool_name, tool_args = parsed
    fn = TOOLS.get(tool_name)
    if fn is None:
        return False, "TOOL_FAIL", [(action_text, None)], time.time() - t0
    result = fn(*tool_args)
    return (result == expected), result, [(action_text, result)], time.time() - t0

for task in TASKS:
    name, args, expected = task
    for r in range(N_RUNS):
        ok, out, path, dt = run_once(act, task)
        rows.append((name, args, r, ok, out, dt, path))

print("task         args     run   pass   output     time(us)")
for name, args, r, ok, out, dt, path in rows:
    print("%-12s %-6s %3d   %s   %5s    %6.0f"
          % (name, str(args), r, "yes" if ok else "no", out, dt * 1e6))


The run result is a table of per-record rows; aggregate statistics compress it into three numbers. The first is pass@1: each task is repeated N_RUNS=3 times, and the fraction that pass is pass@1. Here, of 8 tasks, the product action always uses the wrong tool, all 3 runs fail, pass@1=0; the other tasks all pass, pass@1=1. The name pass@1 comes from giving only one chance: at evaluation time the model takes one shot at each task, and we look at the success fraction, rather than letting it retry until it succeeds. Model output is random; repeating the same task 3 times, passing all 3 is what shows stability.

The second is mean duration: average the duration of all runs to get how long each task takes on average. Toy tasks are nearly instantaneous, so the unit is microseconds; on real tasks this number corresponds to economic value, and also to $M_T$ in the expected-cost formula of section 3.

The third is the action path: each run records the pair "action -> output." Printing a success path next to a failure path shows at once where failure is stuck — here the product task called add instead of mul, which is immediately visible on the path. The value of aggregate statistics is not only computing a total score, but making every failure traceable.

In [ ]:
# Aggregate statistics: pass@1, mean duration, success/failure paths, emit an evaluation report
import numpy as np

def aggregate_report(rows):
    """Summarize per-run records into per-task pass@1 and overall statistics."""
    by_task = {}
    for name, args, r, ok, out, dt, path in rows:
        by_task.setdefault(name, []).append(ok)
    report = {}
    for name, oks in by_task.items():
        report[name] = {"pass_at_1": float(np.mean(oks)), "runs": len(oks)}
    overall = float(np.mean([v["pass_at_1"] for v in report.values()]))
    mean_time = float(np.mean([row[5] for row in rows]))
    return report, overall, mean_time

report, overall, mean_time = aggregate_report(rows)

print("== evaluation report ==")
for name, v in report.items():
    print("%-12s pass@1 = %.2f  (%d runs)" % (name, v["pass_at_1"], v["runs"]))
print("overall pass@1 = %.2f, mean duration %.0f us (toy tasks are nearly instantaneous)"
      % (overall, mean_time * 1e6))

# Take one success record and one failure record, print each action path
expected_map = {}
for name, args, exp in TASKS:
    expected_map.setdefault(name, exp)

ok_row = next(row for row in rows if row[3])
bad_row = next(row for row in rows if not row[3])
print("success path: %s -> %s" % (ok_row[6][0][0], ok_row[4]))
print("failure path: %s -> %s (expected %s)"
      % (bad_row[6][0][0], bad_row[4], expected_map[bad_row[0]]))
print("Key observation: the report exposes the systematic failure on the 'product' task, which is the point of evaluation.")


## Summary

The main points of this lecture:

- [ ] Evaluating an Agent has three parts: task environment, scoring function, and aggregate statistics; the vagueness of each is what makes evaluation hard
- [ ] Multiple-choice scores saturate, are not comparable across benchmarks, and sit far from real tasks; they cannot be translated into "how many hours of work can be finished"
- [ ] METR scales task difficulty by human completion time; the 50% time horizon fitted from a logistic is a shared yardstick of ability
- [ ] The log of the time horizon grows approximately linearly with release date, and the slope converts into "doubling days"
- [ ] GDPval uses real economic tasks and a blind-review win rate, with ties counted as 0.5; an automatic scorer has to be aligned with human–human agreement
- [ ] The expected cost of try-n-then-fix converts win rate into expected time; when the win rate is not high enough, model assistance is slower
- [ ] An LLM judge has position bias; averaging a swapped pair of orders can cancel it
- [ ] Retrieval quality is described by RR, RC, and verifiability; the geometric mean punishes a single short board
- [ ] A minimal harness = a task suite + an agent runner + an aggregate report; get it running before talking about metrics


## Exercises

> You may ask an AI to explain the idea. Do not ask it to finish the exercise for you.


**Exercise 1: fit a 50% time horizon**

The table below is a batch of tasks' human durations and a model's success rates; the true 50% time horizon is 1.0 hour. Fit a logistic with scipy, recover h and beta, and assert that the fitted h is near 1.0 and that the predicted success rate at that duration is near 0.5.

Hint: take the log of task_times before fitting; the objective is the negative log-likelihood.


In [ ]:
# Exercise 1: fit the time horizon
import numpy as np
from scipy.optimize import minimize

task_times = np.array([0.1, 0.3, 0.6, 1.0, 1.5, 2.5, 4.0, 7.0])   # human duration (hours)
success = np.array([0.96, 0.84, 0.72, 0.52, 0.40, 0.28, 0.16, 0.10])

def neg_log_likelihood(params, log_t, y):
    log_h, beta = params
    p = 1.0 / (1.0 + np.exp(-(log_h - log_t) * beta))     # fill in 1: logistic model
    eps = 1e-12
    return -np.sum(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

log_t = np.log(task_times)
res = minimize(neg_log_likelihood, [0.0, 2.0], args=(log_t, success),
               method="L-BFGS-B", bounds=[(-5.0, 5.0), (0.1, 10.0)])
log_h_est, beta_est = res.x
h_est = np.exp(log_h_est)                                  # fill in 2: recover the time horizon

assert abs(h_est - 1.0) < 0.3, "h should be near 1.0 hours"
p_at_true = 1.0 / (1.0 + np.exp(-(log_h_est - np.log(1.0)) * beta_est))
assert abs(p_at_true - 0.5) < 0.1, "success rate at the true horizon should be near 0.5"
print("fitted h = %.2f hours (true 1.0), beta = %.2f" % (h_est, beta_est))
print("predicted success rate at t=1.0h = %.3f" % p_at_true)
print("Passed: fitting success rate together with human duration recovers the model's ability yardstick.")


**Exercise 2: win rate and agreement**

The table below is three-value scores of a model and two human experts on 8 tasks. Compute the model's win rate, the automatic scorer's agreement with humans A_HA, and human–human agreement A_HH, and assert that A_HA is below A_HH.

Hint: the win rate is the mean of the scores (ties counted as 0.5); agreement is the mean of 1 - |x - y|, and in A_HA the human score is the mean of the two experts.


In [ ]:
# Exercise 2: win rate and agreement
import numpy as np

model = np.array([1.0, 0.5, 0.0, 1.0, 0.5, 0.0, 1.0, 0.5])
human1 = np.array([1.0, 0.5, 0.5, 0.5, 0.5, 0.0, 1.0, 1.0])
human2 = np.array([1.0, 1.0, 0.5, 0.5, 0.5, 0.0, 1.0, 0.5])

win_rate = np.mean(model)                              # fill in 1: win rate

def agreement(x, y):
    """Agreement of two score sequences, 1 - |x - y|."""
    return np.mean(1.0 - np.abs(x - y))                # fill in 2

A_HA = agreement(model, (human1 + human2) / 2.0)
A_HH = agreement(human1, human2)

assert abs(win_rate - 0.5625) < 1e-9, "win rate should be 0.5625"
assert A_HA < A_HH, "the automatic scorer should be less consistent than humans with each other"
print("win rate = %.3f" % win_rate)
print("A_HA = %.3f, A_HH = %.3f" % (A_HA, A_HH))
print("Passed: evaluating an automatic scorer uses human–human agreement as the baseline.")


**Exercise 3: nugget coverage**

A human exemplar is made of 5 atomic facts (nuggets); two generated reports are given as lists of sentences. Implement a coverage function that judges whether each nugget appears as a substring in the report, and assert that report B's coverage is higher.

Hint: a nugget is a key point at the atomic-fact level; `n in text` is enough to test the substring.


In [ ]:
# Exercise 3: nugget coverage
import numpy as np

nuggets = ["gravity", "equivalence principle", "light bending", "time dilation", "gravitational waves"]
report_a = ["gravity makes apples fall", "equivalence principle describes local equivalence"]
report_b = ["gravity makes apples fall", "equivalence principle describes local equivalence", "light bending has been observationally confirmed",
            "time dilation is significant in strong fields", "gravitational waves arise from binary mergers"]

def coverage(nuggets, report):
    """Compute the report's coverage fraction of the nugget set."""
    text = "".join(report)
    return np.mean([1.0 if n in text else 0.0 for n in nuggets])   # fill in

cov_a = coverage(nuggets, report_a)
cov_b = coverage(nuggets, report_b)

assert cov_b > cov_a, "report B covers more, so its score should be higher"
print("report A nugget coverage = %.2f" % cov_a)
print("report B nugget coverage = %.2f" % cov_b)
print("Passed: nugget coverage compresses an open-ended survey into an automatically judged recall.")


## References

- METR, [Measuring AI Ability to Complete Long Software Tasks](https://arxiv.org/abs/2503.14499), 2025 — proposes the 50% time-horizon metric, scales Agent ability by human duration, finds that frontier models roughly double every 7 months
- OpenAI, [GDPval: Evaluating AI Model Performance on Real-World Economically Valuable Tasks](https://arxiv.org/abs/2510.04374), 2025 — real economic tasks across 44 occupations in 9 industries; a blind-review win rate shows frontier models approaching human experts
- Patel et al., [DeepScholar-Bench: A Live Benchmark and Automated Evaluation for Generative Research Synthesis](https://arxiv.org/abs/2508.20033), 2025 — three-dimension, 7-metric automatic evaluation of deep research; every system's geometric mean is below 31%
- Wijk et al., [RE-Bench: Evaluating Frontier AI R&D Capabilities of Language Model Agents Against Human Experts](https://arxiv.org/abs/2411.15114), 2024 — 8-hour machine-learning research tasks with a human-expert baseline; one source of the METR task suite
- Panickssery, Bowman & Feng, [LLM Evaluators Recognize and Favor Their Own Generations](https://arxiv.org/abs/2404.13076), 2024 — LLM judges recognize and prefer their own generations; evidence of automatic-scorer bias
- HCAST: Human-Calibrated Autonomy Software Tasks (2025, paper forthcoming) — the main METR task suite, covering ordinary software tasks from 1 minute to 30 hours
- Ngo, [Clarifying and Predicting AGI](https://www.alignmentforum.org/posts/5pRbXo6EEnRBSHokF/clarifying-and-predicting-agi), 2023 — source of the "1-month AGI (167 hours)" definition and of ability-extrapolation thinking
- OpenAI, [evals.openai.com](https://evals.openai.com) — GDPval's 220 gold tasks and the experimental automatic scoring service
- [DeepScholar-Bench official repository](https://github.com/guestrin-lab/deepscholar-bench) — live data pipeline and evaluation code
- Stanford, [CS329A course syllabus](https://cs329a.stanford.edu/) — this lecture's place on the course map
